# Pump Predictive Maintenance Data Science Process

Charter-aligned notebook with data inspection, cleaning, feature engineering, visualization, modelling, alert evaluation, and running-fleet predictions.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "pump_master.csv").exists():
    BASE_DIR = next(
        (parent for parent in [BASE_DIR, *BASE_DIR.parents]
         if (parent / "pump_master.csv").exists()),
        BASE_DIR,
    )

OUTPUT_DIR = BASE_DIR / "datascience_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SENSOR_COLUMNS = [
    "vibration_rms_mm_s",
    "bearing_temp_c",
    "motor_current_a",
    "discharge_pressure_bar",
    "flow_rate_m3h",
    "suction_pressure_bar",
    "speed_rpm",
    "ambient_temp_c",
]

PLANNED_REPAIR_COST = 40_000
BREAKDOWN_COST = 350_000
TARGET_NOTICE_HOURS = 72


## 1. Data Inspection

Inspect schemas, row counts, pump coverage, missing values, duplicates, and maintenance events before cleaning.

In [2]:
pump_master = pd.read_csv(BASE_DIR / "pump_master.csv")
maintenance = pd.read_csv(BASE_DIR / "maintenance_log.csv")
history_raw = pd.read_csv(BASE_DIR / "sensor_history_run_to_failure.csv")
current_raw = pd.read_csv(BASE_DIR / "sensor_current_fleet_no_failure_yet.csv")

inspection_summary = pd.DataFrame({
    "dataset": [
        "pump_master",
        "maintenance_log",
        "run_to_failure_history",
        "current_fleet",
    ],
    "rows": [
        len(pump_master),
        len(maintenance),
        len(history_raw),
        len(current_raw),
    ],
    "unique_pumps": [
        pump_master["pump_id"].nunique(),
        maintenance["pump_id"].nunique(),
        history_raw["pump_id"].nunique(),
        current_raw["pump_id"].nunique(),
    ],
})

print(inspection_summary.to_string(index=False))
print("\nMissing values - run-to-failure:")
print(history_raw.isna().sum().sort_values(ascending=False).to_string())
print("\nMissing values - current fleet:")
print(current_raw.isna().sum().sort_values(ascending=False).to_string())
print("\nDuplicate pump-hour keys:")
print("history:", history_raw.duplicated(["pump_id", "timestamp"]).sum())
print("current:", current_raw.duplicated(["pump_id", "timestamp"]).sum())

maintenance["timestamp"] = pd.to_datetime(
    maintenance["timestamp"], errors="coerce"
)
maintenance["is_breakdown"] = maintenance["activity"].str.startswith(
    "BREAKDOWN", na=False
)
print("\nMaintenance events:")
print(maintenance.groupby(["is_breakdown", "activity"]).size().to_string())


               dataset   rows  unique_pumps
           pump_master    130           130
       maintenance_log    215           116
run_to_failure_history 167584            95
         current_fleet  48857            35

Missing values - run-to-failure:
bearing_temp_c            872
vibration_rms_mm_s        869
pump_id                     0
hour_index                  0
timestamp                   0
motor_current_a             0
discharge_pressure_bar      0
flow_rate_m3h               0
suction_pressure_bar        0
speed_rpm                   0
ambient_temp_c              0

Missing values - current fleet:
bearing_temp_c            103
vibration_rms_mm_s        102
pump_id                     0
hour_index                  0
timestamp                   0
motor_current_a             0
discharge_pressure_bar      0
flow_rate_m3h               0
suction_pressure_bar        0
speed_rpm                   0
ambient_temp_c              0

Duplicate pump-hour keys:
history: 288
current: 192


## 2. Cleaning and Feature Engineering

Deduplicate pump-hour readings, convert sentinels to missing values, interpolate gaps, replace stuck readings, and create health features.

In [3]:
def clean_sensor(path: Path, name: str) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(path)
    initial_rows = len(df)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values(["pump_id", "timestamp", "hour_index"])

    duplicate_timestamp_keys = int(df.duplicated(["pump_id", "timestamp"]).sum())
    duplicate_rows = int(df.duplicated(["pump_id", "timestamp"], keep=False).sum())

    df = (
        df.groupby(["pump_id", "timestamp"], as_index=False)
        .agg({
            "hour_index": "median",
            **{column: "median" for column in SENSOR_COLUMNS},
        })
    )
    df["hour_index"] = df["hour_index"].round().astype("Int64")

    missing_before = df[SENSOR_COLUMNS].isna().sum()
    sentinel_counts = {}
    if "vibration_rms_mm_s" in df:
        sentinel_counts["vibration_99_99"] = int(
            (df["vibration_rms_mm_s"] == 99.99).sum()
        )
        df.loc[df["vibration_rms_mm_s"] == 99.99, "vibration_rms_mm_s"] = np.nan

    df[SENSOR_COLUMNS] = (
        df.groupby("pump_id", group_keys=False)[SENSOR_COLUMNS]
        .apply(lambda group: group.interpolate(method="linear", limit_direction="both"))
    )
    df[SENSOR_COLUMNS] = df[SENSOR_COLUMNS].ffill().bfill()

    stuck_values_replaced = {}
    for column in SENSOR_COLUMNS:
        if df[column].isna().all():
            continue
        rolling_stuck = df.groupby("pump_id")[column].transform(
            lambda series: series.rolling(24, min_periods=24).apply(
                lambda values: 1.0 if np.nanstd(values) == 0 else 0.0,
                raw=True,
            )
        )
        replacements = int((rolling_stuck == 1).sum())
        stuck_values_replaced[column] = replacements
        df.loc[rolling_stuck == 1, column] = np.nan

    df[SENSOR_COLUMNS] = (
        df.groupby("pump_id", group_keys=False)[SENSOR_COLUMNS]
        .apply(lambda group: group.interpolate(method="linear", limit_direction="both"))
    )
    df[SENSOR_COLUMNS] = df[SENSOR_COLUMNS].ffill().bfill()

    df["flow_per_amp"] = df["flow_rate_m3h"] / df["motor_current_a"].replace(0, np.nan)
    df["temp_minus_ambient_c"] = df["bearing_temp_c"] - df["ambient_temp_c"]
    df["pressure_margin_bar"] = (
        df["discharge_pressure_bar"] - df["suction_pressure_bar"]
    )

    health_columns = SENSOR_COLUMNS + [
        "flow_per_amp",
        "temp_minus_ambient_c",
        "pressure_margin_bar",
    ]
    for column in health_columns:
        df[f"{column}_24h_mean"] = df.groupby("pump_id")[column].transform(
            lambda series: series.rolling(24, min_periods=6).mean()
        )
        df[f"{column}_24h_std"] = df.groupby("pump_id")[column].transform(
            lambda series: series.rolling(24, min_periods=6).std()
        )
        df[f"{column}_72h_mean"] = df.groupby("pump_id")[column].transform(
            lambda series: series.rolling(72, min_periods=12).mean()
        )

    for column in ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a",
                   "discharge_pressure_bar", "flow_rate_m3h"]:
        df[f"{column}_24h_slope"] = df.groupby("pump_id")[column].transform(
            lambda series: series.rolling(24, min_periods=12).apply(
                slope,
                raw=True,
            )
        )

    quality = {
        "dataset": name,
        "initial_rows": initial_rows,
        "deduplicated_rows": len(df),
        "rows_removed_by_deduplication": initial_rows - len(df),
        "duplicate_timestamp_keys": duplicate_timestamp_keys,
        "duplicate_rows_including_originals": duplicate_rows,
        "pump_count": int(df["pump_id"].nunique()),
        "timestamp_min": str(df["timestamp"].min()),
        "timestamp_max": str(df["timestamp"].max()),
        "missing_values_before_cleaning": missing_before.to_dict(),
        "missing_values_after_cleaning": df[SENSOR_COLUMNS].isna().sum().to_dict(),
        "sentinel_counts": sentinel_counts,
        "stuck_values_replaced": stuck_values_replaced, 
    }
    return df, quality


In [4]:
def slope(values: np.ndarray) -> float:
    if len(values) < 12:
        return np.nan
    x = np.arange(len(values), dtype=float)
    return float(np.polyfit(x, values, 1)[0])


In [5]:
history, history_quality = clean_sensor(
    BASE_DIR / "sensor_history_run_to_failure.csv",
    "run_to_failure_history",
)
current, current_quality = clean_sensor(
    BASE_DIR / "sensor_current_fleet_no_failure_yet.csv",
    "current_fleet",
)

print(json.dumps({
    "history": history_quality,
    "current": current_quality,
}, indent=2))

print("\nCleaned history shape:", history.shape)
print("Cleaned current shape:", current.shape)


{
  "history": {
    "dataset": "run_to_failure_history",
    "initial_rows": 167584,
    "deduplicated_rows": 167296,
    "rows_removed_by_deduplication": 288,
    "duplicate_timestamp_keys": 288,
    "duplicate_rows_including_originals": 576,
    "pump_count": 95,
    "timestamp_min": "2023-01-02 08:00:00",
    "timestamp_max": "2024-02-08 23:00:00",
    "missing_values_before_cleaning": {
      "vibration_rms_mm_s": 869,
      "bearing_temp_c": 872,
      "motor_current_a": 0,
      "discharge_pressure_bar": 0,
      "flow_rate_m3h": 0,
      "suction_pressure_bar": 0,
      "speed_rpm": 0,
      "ambient_temp_c": 0
    },
    "missing_values_after_cleaning": {
      "vibration_rms_mm_s": 0,
      "bearing_temp_c": 0,
      "motor_current_a": 0,
      "discharge_pressure_bar": 0,
      "flow_rate_m3h": 0,
      "suction_pressure_bar": 0,
      "speed_rpm": 0,
      "ambient_temp_c": 0
    },
    "sentinel_counts": {
      "vibration_99_99": 472
    },
    "stuck_values_replaced": {


## 3. Labels and Derived Features

In [6]:
def asymmetric_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)
    error = y_pred - y_true
    weights = np.where(error > 0, 3.0, 1.0)
    score = float(np.sqrt(np.mean(weights * error ** 2)))
    return rmse, score


In [7]:
def failure_modes(maintenance: pd.DataFrame) -> pd.DataFrame:
    breakdown = maintenance[
        maintenance["activity"].str.startswith("BREAKDOWN", na=False)
    ].copy()
    breakdown["failure_mode"] = (
        breakdown["activity"].str.extract(r"BREAKDOWN\s*-\s*(.*)", expand=False)
    )
    return breakdown[["pump_id", "failure_mode", "timestamp", "downtime_hours"]]


In [8]:
def pump_feature_table(df: pd.DataFrame) -> pd.DataFrame:
    feature_names = [
        "vibration_rms_mm_s",
        "bearing_temp_c",
        "motor_current_a",
        "discharge_pressure_bar",
        "flow_rate_m3h",
        "suction_pressure_bar",
        "speed_rpm",
        "ambient_temp_c",
        "flow_per_amp",
        "temp_minus_ambient_c",
        "pressure_margin_bar",
        "vibration_rms_mm_s_24h_mean",
        "bearing_temp_c_24h_mean",
        "motor_current_a_24h_mean",
        "discharge_pressure_bar_24h_mean",
        "flow_rate_m3h_24h_mean",
        "vibration_rms_mm_s_24h_std",
        "bearing_temp_c_24h_std",
        "motor_current_a_24h_std",
        "flow_rate_m3h_24h_std",
        "vibration_rms_mm_s_24h_slope",
        "bearing_temp_c_24h_slope",
    ]
    available = [column for column in feature_names if column in df]
    rows = []
    for pump_id, group in df.groupby("pump_id"):
        tail = group.tail(72)
        row = {"pump_id": pump_id}
        for column in available:
            row[f"{column}_72h_mean"] = tail[column].mean()
            row[f"{column}_72h_std"] = tail[column].std()
            row[f"{column}_latest"] = tail[column].iloc[-1]
        rows.append(row)
    return pd.DataFrame(rows)


## 4. Modelling and Alert Evaluation

## Visualization

The charts below are generated from cleaned sensor data, maintenance records, RUL predictions, failure-mode predictions, and model comparison results.

In [ ]:
def create_visualizations(
    history: pd.DataFrame,
    current: pd.DataFrame,
    maintenance: pd.DataFrame,
    pump_master: pd.DataFrame,
    rul_results: dict,
    failure_results: dict,
    sequence_results: dict,
) -> list[Path]:
    plots_dir = OUTPUT_DIR / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    created = []

    def save(fig, name: str) -> None:
        path = plots_dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        created.append(path)

    activity = (
        maintenance.assign(
            breakdown=maintenance["activity"].str.startswith("BREAKDOWN", na=False)
        )
        .groupby("activity")
        .agg(events=("pump_id", "size"), downtime=("downtime_hours", "sum"))
        .reset_index()
        .sort_values("events", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=activity,
        x="events",
        y="activity",
        hue="activity",
        palette="tab20",
        legend=False,
        ax=ax,
    )
    ax.set_title("Maintenance and Breakdown Activity Counts")
    ax.set_xlabel("Number of Events")
    ax.set_ylabel("")
    save(fig, "01_maintenance_activity.png")

    pump_lifetimes = (
        history.groupby("pump_id")["hour_index"].max().rename("life_hours")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(pump_lifetimes, bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.axvline(
        pump_lifetimes.mean(),
        color="#E45756",
        linestyle="--",
        label=f"Mean: {pump_lifetimes.mean():.0f} hours",
    )
    ax.set_title("Run-to-Failure Lifetime Distribution")
    ax.set_xlabel("Observed Lifetime (hours)")
    ax.set_ylabel("Pump Count")
    ax.legend()
    save(fig, "02_lifetime_distribution.png")

    trend_columns = ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a", "flow_rate_m3h"]
    for dataset_name, frame in [("history", history), ("current", current)]:
        pumps = list(frame["pump_id"].drop_duplicates())[:10]
        subset = frame[frame["pump_id"].isin(pumps)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        for ax, column in zip(axes.flat, trend_columns):
            sns.lineplot(
                data=subset,
                x="hour_index",
                y=column,
                hue="pump_id",
                estimator=None,
                linewidth=0.8,
                alpha=0.8,
                legend=False,
                ax=ax,
            )
            ax.set_title(column.replace("_", " ").title())
            ax.set_xlabel("Hour Index")
        fig.suptitle(f"Sensor Trends - First 10 {dataset_name.title()} Pumps")
        save(fig, f"03_{dataset_name}_sensor_trends.png")

    breakdown = failure_modes(maintenance)
    history_modes = history.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="left",
    )
    history_modes["life_fraction_elapsed"] = (
        history_modes["hour_index"]
        / history_modes.groupby("pump_id")["hour_index"].transform("max")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, column, title in [
        (axes[0], "vibration_rms_mm_s", "Vibration RMS by Failure Mode"),
        (axes[1], "bearing_temp_c", "Bearing Temperature by Failure Mode"),
    ]:
        sns.lineplot(
            data=history_modes,
            x="life_fraction_elapsed",
            y=column,
            hue="failure_mode",
            estimator="median",
            errorbar=("pi", 50),
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Fraction of Pump Life Elapsed")
    save(fig, "04_normalized_degradation.png")

    correlation_columns = [
        column for column in SENSOR_COLUMNS
        if column in history
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        history[correlation_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Sensor Correlation Matrix")
    save(fig, "05_sensor_correlation.png")

    oof = rul_results["oof_predictions"].copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        oof["rul_hours"],
        oof["predicted_rul_hours"],
        alpha=0.25,
        s=12,
    )
    limit = max(oof["rul_hours"].max(), oof["predicted_rul_hours"].max())
    ax.plot([0, limit], [0, limit], linestyle="--", color="#E45756", label="Perfect Prediction")
    ax.set_xlabel("True RUL (hours)")
    ax.set_ylabel("Predicted RUL (hours)")
    ax.set_title("RUL Out-of-Fold Predictions")
    ax.legend()
    save(fig, "06_rul_prediction_vs_actual.png")

    metric_names = ["RMSE", "Asymmetric"]
    naive_values = [
        rul_results["naive_rmse"],
        rul_results["naive_asymmetric_score"],
    ]
    model_values = [
        rul_results["model_rmse"],
        rul_results["model_asymmetric_score"],
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metric_names))
    ax.bar(x - 0.18, naive_values, width=0.36, label="Naive Baseline", color="#A0CBE8")
    ax.bar(x + 0.18, model_values, width=0.36, label="HistGradientBoosting", color="#4C78A8")
    ax.set_xticks(x, metric_names)
    ax.set_ylabel("Score (lower is better)")
    ax.set_title("RUL Baseline versus Classical Model")
    ax.legend()
    save(fig, "07_rul_model_comparison.png")

    failure_oof = failure_results["oof_predictions"]
    confusion = pd.crosstab(
        failure_oof["actual_failure_mode"],
        failure_oof["predicted_failure_mode"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Predicted Failure Mode")
    ax.set_ylabel("Actual Failure Mode")
    ax.set_title("Failure-Mode Out-of-Fold Recall Matrix")
    save(fig, "08_failure_mode_recall.png")

    current_rul = pd.read_csv(OUTPUT_DIR / "current_fleet_rul_predictions.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=current_rul.sort_values("predicted_rul_hours"),
        x="predicted_rul_hours",
        y="pump_id",
        hue="pump_id",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.axvline(72, color="#E45756", linestyle="--", label="72-hour alert threshold")
    ax.set_xlabel("Predicted RUL (hours)")
    ax.set_ylabel("Pump")
    ax.set_title("Current-Fleet RUL Predictions")
    ax.legend()
    save(fig, "09_current_fleet_rul.png")

    return created


In [ ]:
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualization output folder:", PLOTS_DIR)

## Visualization

The charts below are generated from cleaned sensor data, maintenance records, RUL predictions, failure-mode predictions, and model comparison results.

In [9]:
def create_visualizations(
    history: pd.DataFrame,
    current: pd.DataFrame,
    maintenance: pd.DataFrame,
    pump_master: pd.DataFrame,
    rul_results: dict,
    failure_results: dict,
    sequence_results: dict,
) -> list[Path]:
    plots_dir = OUTPUT_DIR / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    created = []

    def save(fig, name: str) -> None:
        path = plots_dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        created.append(path)

    activity = (
        maintenance.assign(
            breakdown=maintenance["activity"].str.startswith("BREAKDOWN", na=False)
        )
        .groupby("activity")
        .agg(events=("pump_id", "size"), downtime=("downtime_hours", "sum"))
        .reset_index()
        .sort_values("events", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=activity,
        x="events",
        y="activity",
        hue="activity",
        palette="tab20",
        legend=False,
        ax=ax,
    )
    ax.set_title("Maintenance and Breakdown Activity Counts")
    ax.set_xlabel("Number of Events")
    ax.set_ylabel("")
    save(fig, "01_maintenance_activity.png")

    pump_lifetimes = (
        history.groupby("pump_id")["hour_index"].max().rename("life_hours")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(pump_lifetimes, bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.axvline(
        pump_lifetimes.mean(),
        color="#E45756",
        linestyle="--",
        label=f"Mean: {pump_lifetimes.mean():.0f} hours",
    )
    ax.set_title("Run-to-Failure Lifetime Distribution")
    ax.set_xlabel("Observed Lifetime (hours)")
    ax.set_ylabel("Pump Count")
    ax.legend()
    save(fig, "02_lifetime_distribution.png")

    trend_columns = ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a", "flow_rate_m3h"]
    for dataset_name, frame in [("history", history), ("current", current)]:
        pumps = list(frame["pump_id"].drop_duplicates())[:10]
        subset = frame[frame["pump_id"].isin(pumps)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        for ax, column in zip(axes.flat, trend_columns):
            sns.lineplot(
                data=subset,
                x="hour_index",
                y=column,
                hue="pump_id",
                estimator=None,
                linewidth=0.8,
                alpha=0.8,
                legend=False,
                ax=ax,
            )
            ax.set_title(column.replace("_", " ").title())
            ax.set_xlabel("Hour Index")
        fig.suptitle(f"Sensor Trends - First 10 {dataset_name.title()} Pumps")
        save(fig, f"03_{dataset_name}_sensor_trends.png")

    breakdown = failure_modes(maintenance)
    history_modes = history.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="left",
    )
    history_modes["life_fraction_elapsed"] = (
        history_modes["hour_index"]
        / history_modes.groupby("pump_id")["hour_index"].transform("max")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, column, title in [
        (axes[0], "vibration_rms_mm_s", "Vibration RMS by Failure Mode"),
        (axes[1], "bearing_temp_c", "Bearing Temperature by Failure Mode"),
    ]:
        sns.lineplot(
            data=history_modes,
            x="life_fraction_elapsed",
            y=column,
            hue="failure_mode",
            estimator="median",
            errorbar=("pi", 50),
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Fraction of Pump Life Elapsed")
    save(fig, "04_normalized_degradation.png")

    correlation_columns = [
        column for column in SENSOR_COLUMNS
        if column in history
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        history[correlation_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Sensor Correlation Matrix")
    save(fig, "05_sensor_correlation.png")

    oof = rul_results["oof_predictions"].copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        oof["rul_hours"],
        oof["predicted_rul_hours"],
        alpha=0.25,
        s=12,
    )
    limit = max(oof["rul_hours"].max(), oof["predicted_rul_hours"].max())
    ax.plot([0, limit], [0, limit], linestyle="--", color="#E45756", label="Perfect Prediction")
    ax.set_xlabel("True RUL (hours)")
    ax.set_ylabel("Predicted RUL (hours)")
    ax.set_title("RUL Out-of-Fold Predictions")
    ax.legend()
    save(fig, "06_rul_prediction_vs_actual.png")

    metric_names = ["RMSE", "Asymmetric"]
    naive_values = [
        rul_results["naive_rmse"],
        rul_results["naive_asymmetric_score"],
    ]
    model_values = [
        rul_results["model_rmse"],
        rul_results["model_asymmetric_score"],
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metric_names))
    ax.bar(x - 0.18, naive_values, width=0.36, label="Naive Baseline", color="#A0CBE8")
    ax.bar(x + 0.18, model_values, width=0.36, label="HistGradientBoosting", color="#4C78A8")
    ax.set_xticks(x, metric_names)
    ax.set_ylabel("Score (lower is better)")
    ax.set_title("RUL Baseline versus Classical Model")
    ax.legend()
    save(fig, "07_rul_model_comparison.png")

    failure_oof = failure_results["oof_predictions"]
    confusion = pd.crosstab(
        failure_oof["actual_failure_mode"],
        failure_oof["predicted_failure_mode"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Predicted Failure Mode")
    ax.set_ylabel("Actual Failure Mode")
    ax.set_title("Failure-Mode Out-of-Fold Recall Matrix")
    save(fig, "08_failure_mode_recall.png")

    current_rul = pd.read_csv(OUTPUT_DIR / "current_fleet_rul_predictions.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=current_rul.sort_values("predicted_rul_hours"),
        x="predicted_rul_hours",
        y="pump_id",
        hue="pump_id",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.axvline(72, color="#E45756", linestyle="--", label="72-hour alert threshold")
    ax.set_xlabel("Predicted RUL (hours)")
    ax.set_ylabel("Pump")
    ax.set_title("Current-Fleet RUL Predictions")
    ax.legend()
    save(fig, "09_current_fleet_rul.png")

    return created


In [10]:
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualization output folder:", PLOTS_DIR)

Visualization output folder: d:\Predictive-Maintenance-Remaining-Useful-Life-of-Industrial-Pumps-main\datascience_outputs\plots


## Visualization

The charts below are generated from cleaned sensor data, maintenance records, RUL predictions, failure-mode predictions, and model comparison results.

In [11]:
def create_visualizations(
    history: pd.DataFrame,
    current: pd.DataFrame,
    maintenance: pd.DataFrame,
    pump_master: pd.DataFrame,
    rul_results: dict,
    failure_results: dict,
    sequence_results: dict,
) -> list[Path]:
    plots_dir = OUTPUT_DIR / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    created = []

    def save(fig, name: str) -> None:
        path = plots_dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        created.append(path)

    activity = (
        maintenance.assign(
            breakdown=maintenance["activity"].str.startswith("BREAKDOWN", na=False)
        )
        .groupby("activity")
        .agg(events=("pump_id", "size"), downtime=("downtime_hours", "sum"))
        .reset_index()
        .sort_values("events", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=activity,
        x="events",
        y="activity",
        hue="activity",
        palette="tab20",
        legend=False,
        ax=ax,
    )
    ax.set_title("Maintenance and Breakdown Activity Counts")
    ax.set_xlabel("Number of Events")
    ax.set_ylabel("")
    save(fig, "01_maintenance_activity.png")

    pump_lifetimes = (
        history.groupby("pump_id")["hour_index"].max().rename("life_hours")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(pump_lifetimes, bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.axvline(
        pump_lifetimes.mean(),
        color="#E45756",
        linestyle="--",
        label=f"Mean: {pump_lifetimes.mean():.0f} hours",
    )
    ax.set_title("Run-to-Failure Lifetime Distribution")
    ax.set_xlabel("Observed Lifetime (hours)")
    ax.set_ylabel("Pump Count")
    ax.legend()
    save(fig, "02_lifetime_distribution.png")

    trend_columns = ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a", "flow_rate_m3h"]
    for dataset_name, frame in [("history", history), ("current", current)]:
        pumps = list(frame["pump_id"].drop_duplicates())[:10]
        subset = frame[frame["pump_id"].isin(pumps)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        for ax, column in zip(axes.flat, trend_columns):
            sns.lineplot(
                data=subset,
                x="hour_index",
                y=column,
                hue="pump_id",
                estimator=None,
                linewidth=0.8,
                alpha=0.8,
                legend=False,
                ax=ax,
            )
            ax.set_title(column.replace("_", " ").title())
            ax.set_xlabel("Hour Index")
        fig.suptitle(f"Sensor Trends - First 10 {dataset_name.title()} Pumps")
        save(fig, f"03_{dataset_name}_sensor_trends.png")

    breakdown = failure_modes(maintenance)
    history_modes = history.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="left",
    )
    history_modes["life_fraction_elapsed"] = (
        history_modes["hour_index"]
        / history_modes.groupby("pump_id")["hour_index"].transform("max")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, column, title in [
        (axes[0], "vibration_rms_mm_s", "Vibration RMS by Failure Mode"),
        (axes[1], "bearing_temp_c", "Bearing Temperature by Failure Mode"),
    ]:
        sns.lineplot(
            data=history_modes,
            x="life_fraction_elapsed",
            y=column,
            hue="failure_mode",
            estimator="median",
            errorbar=("pi", 50),
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Fraction of Pump Life Elapsed")
    save(fig, "04_normalized_degradation.png")

    correlation_columns = [
        column for column in SENSOR_COLUMNS
        if column in history
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        history[correlation_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Sensor Correlation Matrix")
    save(fig, "05_sensor_correlation.png")

    oof = rul_results["oof_predictions"].copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        oof["rul_hours"],
        oof["predicted_rul_hours"],
        alpha=0.25,
        s=12,
    )
    limit = max(oof["rul_hours"].max(), oof["predicted_rul_hours"].max())
    ax.plot([0, limit], [0, limit], linestyle="--", color="#E45756", label="Perfect Prediction")
    ax.set_xlabel("True RUL (hours)")
    ax.set_ylabel("Predicted RUL (hours)")
    ax.set_title("RUL Out-of-Fold Predictions")
    ax.legend()
    save(fig, "06_rul_prediction_vs_actual.png")

    metric_names = ["RMSE", "Asymmetric"]
    naive_values = [
        rul_results["naive_rmse"],
        rul_results["naive_asymmetric_score"],
    ]
    model_values = [
        rul_results["model_rmse"],
        rul_results["model_asymmetric_score"],
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metric_names))
    ax.bar(x - 0.18, naive_values, width=0.36, label="Naive Baseline", color="#A0CBE8")
    ax.bar(x + 0.18, model_values, width=0.36, label="HistGradientBoosting", color="#4C78A8")
    ax.set_xticks(x, metric_names)
    ax.set_ylabel("Score (lower is better)")
    ax.set_title("RUL Baseline versus Classical Model")
    ax.legend()
    save(fig, "07_rul_model_comparison.png")

    failure_oof = failure_results["oof_predictions"]
    confusion = pd.crosstab(
        failure_oof["actual_failure_mode"],
        failure_oof["predicted_failure_mode"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Predicted Failure Mode")
    ax.set_ylabel("Actual Failure Mode")
    ax.set_title("Failure-Mode Out-of-Fold Recall Matrix")
    save(fig, "08_failure_mode_recall.png")

    current_rul = pd.read_csv(OUTPUT_DIR / "current_fleet_rul_predictions.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=current_rul.sort_values("predicted_rul_hours"),
        x="predicted_rul_hours",
        y="pump_id",
        hue="pump_id",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.axvline(72, color="#E45756", linestyle="--", label="72-hour alert threshold")
    ax.set_xlabel("Predicted RUL (hours)")
    ax.set_ylabel("Pump")
    ax.set_title("Current-Fleet RUL Predictions")
    ax.legend()
    save(fig, "09_current_fleet_rul.png")

    return created


In [12]:
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualization output folder:", PLOTS_DIR)

Visualization output folder: d:\Predictive-Maintenance-Remaining-Useful-Life-of-Industrial-Pumps-main\datascience_outputs\plots


## Visualization

The charts below are generated from cleaned sensor data, maintenance records, RUL predictions, failure-mode predictions, and model comparison results.

In [13]:
def create_visualizations(
    history: pd.DataFrame,
    current: pd.DataFrame,
    maintenance: pd.DataFrame,
    pump_master: pd.DataFrame,
    rul_results: dict,
    failure_results: dict,
    sequence_results: dict,
) -> list[Path]:
    plots_dir = OUTPUT_DIR / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    created = []

    def save(fig, name: str) -> None:
        path = plots_dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        created.append(path)

    activity = (
        maintenance.assign(
            breakdown=maintenance["activity"].str.startswith("BREAKDOWN", na=False)
        )
        .groupby("activity")
        .agg(events=("pump_id", "size"), downtime=("downtime_hours", "sum"))
        .reset_index()
        .sort_values("events", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=activity,
        x="events",
        y="activity",
        hue="activity",
        palette="tab20",
        legend=False,
        ax=ax,
    )
    ax.set_title("Maintenance and Breakdown Activity Counts")
    ax.set_xlabel("Number of Events")
    ax.set_ylabel("")
    save(fig, "01_maintenance_activity.png")

    pump_lifetimes = (
        history.groupby("pump_id")["hour_index"].max().rename("life_hours")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(pump_lifetimes, bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.axvline(
        pump_lifetimes.mean(),
        color="#E45756",
        linestyle="--",
        label=f"Mean: {pump_lifetimes.mean():.0f} hours",
    )
    ax.set_title("Run-to-Failure Lifetime Distribution")
    ax.set_xlabel("Observed Lifetime (hours)")
    ax.set_ylabel("Pump Count")
    ax.legend()
    save(fig, "02_lifetime_distribution.png")

    trend_columns = ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a", "flow_rate_m3h"]
    for dataset_name, frame in [("history", history), ("current", current)]:
        pumps = list(frame["pump_id"].drop_duplicates())[:10]
        subset = frame[frame["pump_id"].isin(pumps)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        for ax, column in zip(axes.flat, trend_columns):
            sns.lineplot(
                data=subset,
                x="hour_index",
                y=column,
                hue="pump_id",
                estimator=None,
                linewidth=0.8,
                alpha=0.8,
                legend=False,
                ax=ax,
            )
            ax.set_title(column.replace("_", " ").title())
            ax.set_xlabel("Hour Index")
        fig.suptitle(f"Sensor Trends - First 10 {dataset_name.title()} Pumps")
        save(fig, f"03_{dataset_name}_sensor_trends.png")

    breakdown = failure_modes(maintenance)
    history_modes = history.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="left",
    )
    history_modes["life_fraction_elapsed"] = (
        history_modes["hour_index"]
        / history_modes.groupby("pump_id")["hour_index"].transform("max")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, column, title in [
        (axes[0], "vibration_rms_mm_s", "Vibration RMS by Failure Mode"),
        (axes[1], "bearing_temp_c", "Bearing Temperature by Failure Mode"),
    ]:
        sns.lineplot(
            data=history_modes,
            x="life_fraction_elapsed",
            y=column,
            hue="failure_mode",
            estimator="median",
            errorbar=("pi", 50),
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Fraction of Pump Life Elapsed")
    save(fig, "04_normalized_degradation.png")

    correlation_columns = [
        column for column in SENSOR_COLUMNS
        if column in history
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        history[correlation_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Sensor Correlation Matrix")
    save(fig, "05_sensor_correlation.png")

    oof = rul_results["oof_predictions"].copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        oof["rul_hours"],
        oof["predicted_rul_hours"],
        alpha=0.25,
        s=12,
    )
    limit = max(oof["rul_hours"].max(), oof["predicted_rul_hours"].max())
    ax.plot([0, limit], [0, limit], linestyle="--", color="#E45756", label="Perfect Prediction")
    ax.set_xlabel("True RUL (hours)")
    ax.set_ylabel("Predicted RUL (hours)")
    ax.set_title("RUL Out-of-Fold Predictions")
    ax.legend()
    save(fig, "06_rul_prediction_vs_actual.png")

    metric_names = ["RMSE", "Asymmetric"]
    naive_values = [
        rul_results["naive_rmse"],
        rul_results["naive_asymmetric_score"],
    ]
    model_values = [
        rul_results["model_rmse"],
        rul_results["model_asymmetric_score"],
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metric_names))
    ax.bar(x - 0.18, naive_values, width=0.36, label="Naive Baseline", color="#A0CBE8")
    ax.bar(x + 0.18, model_values, width=0.36, label="HistGradientBoosting", color="#4C78A8")
    ax.set_xticks(x, metric_names)
    ax.set_ylabel("Score (lower is better)")
    ax.set_title("RUL Baseline versus Classical Model")
    ax.legend()
    save(fig, "07_rul_model_comparison.png")

    failure_oof = failure_results["oof_predictions"]
    confusion = pd.crosstab(
        failure_oof["actual_failure_mode"],
        failure_oof["predicted_failure_mode"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Predicted Failure Mode")
    ax.set_ylabel("Actual Failure Mode")
    ax.set_title("Failure-Mode Out-of-Fold Recall Matrix")
    save(fig, "08_failure_mode_recall.png")

    current_rul = pd.read_csv(OUTPUT_DIR / "current_fleet_rul_predictions.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=current_rul.sort_values("predicted_rul_hours"),
        x="predicted_rul_hours",
        y="pump_id",
        hue="pump_id",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.axvline(72, color="#E45756", linestyle="--", label="72-hour alert threshold")
    ax.set_xlabel("Predicted RUL (hours)")
    ax.set_ylabel("Pump")
    ax.set_title("Current-Fleet RUL Predictions")
    ax.legend()
    save(fig, "09_current_fleet_rul.png")

    return created


In [14]:
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualization output folder:", PLOTS_DIR)

Visualization output folder: d:\Predictive-Maintenance-Remaining-Useful-Life-of-Industrial-Pumps-main\datascience_outputs\plots


## Visualization

The charts below are generated from cleaned sensor data, maintenance records, RUL predictions, failure-mode predictions, and model comparison results.

In [15]:
def create_visualizations(
    history: pd.DataFrame,
    current: pd.DataFrame,
    maintenance: pd.DataFrame,
    pump_master: pd.DataFrame,
    rul_results: dict,
    failure_results: dict,
    sequence_results: dict,
) -> list[Path]:
    plots_dir = OUTPUT_DIR / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    created = []

    def save(fig, name: str) -> None:
        path = plots_dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        created.append(path)

    activity = (
        maintenance.assign(
            breakdown=maintenance["activity"].str.startswith("BREAKDOWN", na=False)
        )
        .groupby("activity")
        .agg(events=("pump_id", "size"), downtime=("downtime_hours", "sum"))
        .reset_index()
        .sort_values("events", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=activity,
        x="events",
        y="activity",
        hue="activity",
        palette="tab20",
        legend=False,
        ax=ax,
    )
    ax.set_title("Maintenance and Breakdown Activity Counts")
    ax.set_xlabel("Number of Events")
    ax.set_ylabel("")
    save(fig, "01_maintenance_activity.png")

    pump_lifetimes = (
        history.groupby("pump_id")["hour_index"].max().rename("life_hours")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(pump_lifetimes, bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.axvline(
        pump_lifetimes.mean(),
        color="#E45756",
        linestyle="--",
        label=f"Mean: {pump_lifetimes.mean():.0f} hours",
    )
    ax.set_title("Run-to-Failure Lifetime Distribution")
    ax.set_xlabel("Observed Lifetime (hours)")
    ax.set_ylabel("Pump Count")
    ax.legend()
    save(fig, "02_lifetime_distribution.png")

    trend_columns = ["vibration_rms_mm_s", "bearing_temp_c", "motor_current_a", "flow_rate_m3h"]
    for dataset_name, frame in [("history", history), ("current", current)]:
        pumps = list(frame["pump_id"].drop_duplicates())[:10]
        subset = frame[frame["pump_id"].isin(pumps)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        for ax, column in zip(axes.flat, trend_columns):
            sns.lineplot(
                data=subset,
                x="hour_index",
                y=column,
                hue="pump_id",
                estimator=None,
                linewidth=0.8,
                alpha=0.8,
                legend=False,
                ax=ax,
            )
            ax.set_title(column.replace("_", " ").title())
            ax.set_xlabel("Hour Index")
        fig.suptitle(f"Sensor Trends - First 10 {dataset_name.title()} Pumps")
        save(fig, f"03_{dataset_name}_sensor_trends.png")

    breakdown = failure_modes(maintenance)
    history_modes = history.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="left",
    )
    history_modes["life_fraction_elapsed"] = (
        history_modes["hour_index"]
        / history_modes.groupby("pump_id")["hour_index"].transform("max")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, column, title in [
        (axes[0], "vibration_rms_mm_s", "Vibration RMS by Failure Mode"),
        (axes[1], "bearing_temp_c", "Bearing Temperature by Failure Mode"),
    ]:
        sns.lineplot(
            data=history_modes,
            x="life_fraction_elapsed",
            y=column,
            hue="failure_mode",
            estimator="median",
            errorbar=("pi", 50),
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Fraction of Pump Life Elapsed")
    save(fig, "04_normalized_degradation.png")

    correlation_columns = [
        column for column in SENSOR_COLUMNS
        if column in history
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        history[correlation_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Sensor Correlation Matrix")
    save(fig, "05_sensor_correlation.png")

    oof = rul_results["oof_predictions"].copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        oof["rul_hours"],
        oof["predicted_rul_hours"],
        alpha=0.25,
        s=12,
    )
    limit = max(oof["rul_hours"].max(), oof["predicted_rul_hours"].max())
    ax.plot([0, limit], [0, limit], linestyle="--", color="#E45756", label="Perfect Prediction")
    ax.set_xlabel("True RUL (hours)")
    ax.set_ylabel("Predicted RUL (hours)")
    ax.set_title("RUL Out-of-Fold Predictions")
    ax.legend()
    save(fig, "06_rul_prediction_vs_actual.png")

    metric_names = ["RMSE", "Asymmetric"]
    naive_values = [
        rul_results["naive_rmse"],
        rul_results["naive_asymmetric_score"],
    ]
    model_values = [
        rul_results["model_rmse"],
        rul_results["model_asymmetric_score"],
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metric_names))
    ax.bar(x - 0.18, naive_values, width=0.36, label="Naive Baseline", color="#A0CBE8")
    ax.bar(x + 0.18, model_values, width=0.36, label="HistGradientBoosting", color="#4C78A8")
    ax.set_xticks(x, metric_names)
    ax.set_ylabel("Score (lower is better)")
    ax.set_title("RUL Baseline versus Classical Model")
    ax.legend()
    save(fig, "07_rul_model_comparison.png")

    failure_oof = failure_results["oof_predictions"]
    confusion = pd.crosstab(
        failure_oof["actual_failure_mode"],
        failure_oof["predicted_failure_mode"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Predicted Failure Mode")
    ax.set_ylabel("Actual Failure Mode")
    ax.set_title("Failure-Mode Out-of-Fold Recall Matrix")
    save(fig, "08_failure_mode_recall.png")

    current_rul = pd.read_csv(OUTPUT_DIR / "current_fleet_rul_predictions.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        data=current_rul.sort_values("predicted_rul_hours"),
        x="predicted_rul_hours",
        y="pump_id",
        hue="pump_id",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.axvline(72, color="#E45756", linestyle="--", label="72-hour alert threshold")
    ax.set_xlabel("Predicted RUL (hours)")
    ax.set_ylabel("Pump")
    ax.set_title("Current-Fleet RUL Predictions")
    ax.legend()
    save(fig, "09_current_fleet_rul.png")

    return created


In [16]:
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("Visualization output folder:", PLOTS_DIR)

Visualization output folder: d:\Predictive-Maintenance-Remaining-Useful-Life-of-Industrial-Pumps-main\datascience_outputs\plots


In [17]:
def model_rul(history: pd.DataFrame, pump_master: pd.DataFrame) -> dict:
    history = history.merge(
        pump_master[["pump_id", "site", "model", "rated_kw", "install_year"]],
        on="pump_id",
        how="left",
    )
    history["rul_hours"] = (
        history.groupby("pump_id")["hour_index"].transform("max")
        - history["hour_index"]
    )

    static_model = pd.get_dummies(
        history[["pump_id", "site", "model"]].drop(columns=["pump_id"]),
        columns=["site", "model"],
    )
    feature_columns = [
        "hour_index",
        "rated_kw",
        "install_year",
        "vibration_rms_mm_s",
        "bearing_temp_c",
        "motor_current_a",
        "discharge_pressure_bar",
        "flow_rate_m3h",
        "suction_pressure_bar",
        "speed_rpm",
        "ambient_temp_c",
        "flow_per_amp",
        "temp_minus_ambient_c",
        "pressure_margin_bar",
        "vibration_rms_mm_s_24h_mean",
        "vibration_rms_mm_s_24h_std",
        "vibration_rms_mm_s_24h_slope",
        "bearing_temp_c_24h_mean",
        "bearing_temp_c_24h_std",
        "bearing_temp_c_24h_slope",
        "motor_current_a_24h_mean",
        "motor_current_a_24h_std",
        "motor_current_a_24h_slope",
        "flow_rate_m3h_24h_mean",
        "flow_rate_m3h_24h_std",
    ]
    feature_columns = [column for column in feature_columns if column in history]
    history = history.dropna(subset=feature_columns + ["rul_hours"])
    X = pd.concat([
        history[feature_columns].reset_index(drop=True),
        static_model.loc[history.index].reset_index(drop=True),
    ], axis=1)
    y = history["rul_hours"].astype(float).to_numpy()
    pump_ids = history["pump_id"].to_numpy()

    groups = np.unique(pump_ids)
    lifetimes = history.groupby("pump_id")["rul_hours"].max()
    mean_lifetime = float(lifetimes.mean())

    fold_predictions = np.zeros(len(y))
    fold_truth = y.copy()
    fold_naive = np.zeros(len(y))
    folds = GroupKFold(n_splits=5)

    for fold, (train_indices, test_indices) in enumerate(
        folds.split(X, y, groups=pump_ids), start=1
    ):
        train_pumps = set(pump_ids[train_indices])
        train_lifetime_mean = float(
            history.iloc[train_indices].groupby("pump_id")["rul_hours"].max().mean()
        )
        model = make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                learning_rate=0.08,
                max_iter=250,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42 + fold,
            ),
        )
        model.fit(X.iloc[train_indices], y[train_indices])
        fold_predictions[test_indices] = model.predict(X.iloc[test_indices])
        fold_naive[test_indices] = train_lifetime_mean
        _ = train_pumps

    fold_predictions = np.clip(fold_predictions, 0, None)
    fold_naive = np.clip(fold_naive, 0, None)
    model_rmse, model_asymmetric = asymmetric_rmse(fold_truth, fold_predictions)
    naive_rmse, naive_asymmetric = asymmetric_rmse(fold_truth, fold_naive)

    alert_results = []
    oof = pd.DataFrame({
        "pump_id": pump_ids,
        "rul_hours": fold_truth,
        "predicted_rul_hours": fold_predictions,
    })
    for pump_id, pump_oof in oof.groupby("pump_id"):
        pump_oof = pump_oof.sort_index()
        alerted_rows = pump_oof[pump_oof["predicted_rul_hours"] <= TARGET_NOTICE_HOURS]
        if alerted_rows.empty:
            alert_row = None
            advance_hours = -1
        else:
            alert_row = alerted_rows.iloc[0]
            advance_hours = float(alert_row["rul_hours"])
        alert_results.append({
            "pump_id": pump_id,
            "alerted": alert_row is not None,
            "advance_hours": advance_hours,
            "alerts_72h_advance": bool(advance_hours >= TARGET_NOTICE_HOURS),
        })
    alert_table = pd.DataFrame(alert_results)
    failure_alert_coverage = float(alert_table["alerts_72h_advance"].mean())

    pump_count = alert_table["pump_id"].nunique()
    alerted_count = int(alert_table["alerted"].sum())
    false_alert_count = int((~alert_table["alerts_72h_advance"] & alert_table["alerted"]).sum())
    total_observed_months = float(
        history.groupby("pump_id")["hour_index"].max().sum() / (30 * 24)
    )
    false_alerts_per_pump_month = false_alert_count / total_observed_months

    failures_with_advance_alert = int(alert_table["alerts_72h_advance"].sum())
    breakdowns_avoided_cost = failures_with_advance_alert * (BREAKDOWN_COST - PLANNED_REPAIR_COST)
    planned_repair_cost = alerted_count * PLANNED_REPAIR_COST
    net_savings = breakdowns_avoided_cost - planned_repair_cost

    full_model = make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingRegressor(
            learning_rate=0.08,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        ),
    )
    full_model.fit(X, y)

    return {
        "mean_lifetime_hours": mean_lifetime,
        "naive_rmse": naive_rmse,
        "naive_asymmetric_score": naive_asymmetric,
        "model_rmse": model_rmse,
        "model_asymmetric_score": model_asymmetric,
        "rmse_improvement_pct": 100 * (naive_rmse - model_rmse) / naive_rmse,
        "asymmetric_improvement_pct": 100 * (naive_asymmetric - model_asymmetric) / naive_asymmetric,
        "failure_alert_coverage_pct": 100 * failure_alert_coverage,
        "false_alert_count": false_alert_count,
        "false_alerts_per_pump_month": false_alerts_per_pump_month,
        "alerted_pumps": alerted_count,
        "pump_count": pump_count,
        "gross_breakdown_cost_avoided_inr": breakdowns_avoided_cost,
        "planned_repair_cost_inr": planned_repair_cost,
        "net_savings_inr": net_savings,
        "oof_predictions": oof,
        "alert_table": alert_table,
        "full_model": full_model,
        "feature_columns": feature_columns,
        "static_columns": static_model.columns.tolist(),
        "model_columns": X.columns.tolist(),
        "history_labeled": history,
    }


In [18]:
def model_failure_mode(history: pd.DataFrame, maintenance: pd.DataFrame,
                       pump_master: pd.DataFrame) -> dict:
    breakdown = failure_modes(maintenance)
    pump_features = pump_feature_table(history)
    pump_features = pump_features.merge(
        pump_master[["pump_id", "site", "model", "rated_kw", "install_year"]],
        on="pump_id",
        how="left",
    )
    pump_features = pump_features.merge(
        breakdown[["pump_id", "failure_mode"]],
        on="pump_id",
        how="inner",
    )

    X = pump_features.drop(columns=["pump_id", "failure_mode"])
    X = pd.get_dummies(X, columns=["site", "model"], dummy_na=False)
    y = pump_features["failure_mode"].to_numpy()
    pump_ids = pump_features["pump_id"].to_numpy()

    predictions = np.empty(len(y), dtype=object)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_indices, test_indices) in enumerate(
        folds.split(X, y), start=1
    ):
        model = make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42 + fold,
                n_jobs=-1,
            ),
        )
        model.fit(X.iloc[train_indices], y[train_indices])
        predictions[test_indices] = model.predict(X.iloc[test_indices])

    accuracy = float(accuracy_score(y, predictions))
    per_mode = {}
    for mode in sorted(set(y)):
        mask = y == mode
        per_mode[mode] = float(accuracy_score(y[mask], predictions[mask]))

    full_model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    )
    full_model.fit(X, y)

    oof = pd.DataFrame({
        "pump_id": pump_ids,
        "actual_failure_mode": y,
        "predicted_failure_mode": predictions,
    })
    return {
        "accuracy": accuracy,
        "per_mode_recall": per_mode,
        "meets_85_percent": accuracy >= 0.85,
        "oof_predictions": oof,
        "full_model": full_model,
        "feature_columns": X.columns.tolist(),
    }


In [19]:
def sequence_cnn(history: pd.DataFrame) -> dict:
    try:
        import tensorflow as tf
        from tensorflow import keras
    except Exception as exc:
        return {"status": "unavailable", "reason": str(exc)}

    tf.keras.utils.set_random_seed(42)
    sequence_columns = SENSOR_COLUMNS
    window = 72
    sequences = []
    labels = []
    pump_ids = []
    for pump_id, group in history.groupby("pump_id"):
        group = group.sort_values("hour_index")
        values = group[sequence_columns].to_numpy(dtype=np.float32)
        max_hour = float(group["hour_index"].max())
        for end in range(window, len(group) + 1, 24):
            sequences.append(values[end - window:end])
            labels.append(max_hour - float(group.iloc[end - 1]["hour_index"]))
            pump_ids.append(pump_id)

    X = np.asarray(sequences)
    y = np.asarray(labels)
    pump_ids = np.asarray(pump_ids)
    unique_pumps = np.unique(pump_ids)
    if len(unique_pumps) < 5:
        return {"status": "unavailable", "reason": "Fewer than five pumps"}

    oof_predictions = np.zeros(len(y))
    for fold, (train_pumps, test_pumps) in enumerate(
        GroupKFold(n_splits=3).split(X, y, groups=pump_ids), start=1
    ):
        mean = X[train_pumps].mean(axis=(0, 1), keepdims=True)
        std = X[train_pumps].std(axis=(0, 1), keepdims=True)
        std = np.where(std < 1e-6, 1.0, std)
        X_train = (X[train_pumps] - mean) / std
        X_test = (X[test_pumps] - mean) / std
        y_mean = y[train_pumps].mean()
        y_std = y[train_pumps].std()
        y_train = (y[train_pumps] - y_mean) / y_std

        def asymmetric_loss(y_true, y_pred):
            error = y_pred - y_true
            weight = tf.where(error > 0, 3.0, 1.0)
            return tf.reduce_mean(weight * tf.square(error))

        model = keras.Sequential([
            keras.layers.Input(shape=(window, len(sequence_columns))),
            keras.layers.Conv1D(32, kernel_size=7, activation="relu"),
            keras.layers.MaxPooling1D(2),
            keras.layers.Conv1D(16, kernel_size=5, activation="relu"),
            keras.layers.GlobalAveragePooling1D(),
            keras.layers.Dense(16, activation="relu"),
            keras.layers.Dense(1),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss=asymmetric_loss,
        )
        model.fit(
            X_train,
            y_train,
            epochs=3,
            batch_size=256,
            verbose=0,
            validation_split=0.1,
        )
        predicted = model.predict(X_test, verbose=0).reshape(-1)
        oof_predictions[test_pumps] = np.clip(predicted * y_std + y_mean, 0, None)

    rmse, asymmetric = asymmetric_rmse(y, oof_predictions)
    return {
        "status": "completed",
        "window_hours": window,
        "folds": 3,
        "sequence_samples": len(y),
        "rmse": rmse,
        "asymmetric_score": asymmetric,
    }


## 5. Run the Complete Pipeline

In [ ]:
def main() -> None:
    pump_master = pd.read_csv(BASE_DIR / "pump_master.csv")
    maintenance = pd.read_csv(BASE_DIR / "maintenance_log.csv")
    maintenance["timestamp"] = pd.to_datetime(maintenance["timestamp"], errors="coerce")

    history, history_quality = clean_sensor(
        BASE_DIR / "sensor_history_run_to_failure.csv",
        "run_to_failure_history",
    )
    current, current_quality = clean_sensor(
        BASE_DIR / "sensor_current_fleet_no_failure_yet.csv",
        "current_fleet",
    )

    history_lifetime = history.groupby("pump_id")["hour_index"].max()
    history_lifetime.to_csv(
        OUTPUT_DIR / "observed_lifetime_by_pump.csv",
        header=["observed_lifetime_hours"],
    )

    maintenance_summary = (
        maintenance.groupby("activity", dropna=False)
        .agg(
            events=("pump_id", "size"),
            pumps=("pump_id", "nunique"),
            total_downtime_hours=("downtime_hours", "sum"),
            mean_downtime_hours=("downtime_hours", "mean"),
        )
        .reset_index()
        .sort_values("events", ascending=False)
    )
    maintenance_summary.to_csv(
        OUTPUT_DIR / "maintenance_activity_summary.csv",
        index=False,
    )

    breakdown_summary = (
        failure_modes(maintenance)
        .groupby("failure_mode")
        .agg(
            events=("pump_id", "size"),
            pumps=("pump_id", "nunique"),
            total_downtime_hours=("downtime_hours", "sum"),
        )
        .reset_index()
        .sort_values("events", ascending=False)
    )
    breakdown_summary.to_csv(
        OUTPUT_DIR / "failure_mode_summary.csv",
        index=False,
    )

    rul_results = model_rul(history, pump_master)
    rul_results["oof_predictions"].to_csv(
        OUTPUT_DIR / "rul_out_of_fold_predictions.csv",
        index=False,
    )
    rul_results["alert_table"].to_csv(
        OUTPUT_DIR / "rul_alert_evaluation.csv",
        index=False,
    )

    failure_results = model_failure_mode(history, maintenance, pump_master)
    failure_results["oof_predictions"].to_csv(
        OUTPUT_DIR / "failure_mode_out_of_fold_predictions.csv",
        index=False,
    )

    sequence_results = sequence_cnn(history)

    current_latest = (
        current.merge(
            pump_master[["pump_id", "site", "model", "rated_kw", "install_year"]],
            on="pump_id",
            how="left",
        )
        .sort_values(["pump_id", "timestamp"])
        .groupby("pump_id", as_index=False)
        .tail(1)
    )
    rul_feature_columns = rul_results["feature_columns"]
    static_columns = rul_results["static_columns"]
    rul_X = current_latest.reindex(columns=rul_feature_columns).copy()
    rul_static = pd.get_dummies(
        current_latest[["site", "model"]],
        columns=["site", "model"],
    )
    for column in static_columns:
        if column not in rul_static:
            rul_static[column] = 0
    rul_static = rul_static[static_columns]
    rul_current_X = pd.concat([
        rul_X.reset_index(drop=True),
        rul_static.reset_index(drop=True),
    ], axis=1).reindex(columns=rul_results["model_columns"])



    current_rul_predictions = np.clip(
        rul_results["full_model"].predict(rul_current_X),
        0,
        None,
    )
    current_rul_output = pd.DataFrame({
        "pump_id": current_latest["pump_id"].to_numpy(),
        "predicted_rul_hours": current_rul_predictions,
        "predicted_rul_days": current_rul_predictions / 24,
        "alert_72h": current_rul_predictions <= TARGET_NOTICE_HOURS,
    })
    current_rul_output.to_csv(
        OUTPUT_DIR / "current_fleet_rul_predictions.csv",
        index=False,
    )

    current_failure_features = pump_feature_table(current)
    current_failure_features = current_failure_features.merge(
        pump_master[["pump_id", "site", "model", "rated_kw", "install_year"]],
        on="pump_id",
        how="left",
    )
    current_failure_X = pd.get_dummies(
        current_failure_features.drop(columns=["pump_id"]),
        columns=["site", "model"],
        dummy_na=False,
    )
    for column in failure_results["feature_columns"]:
        if column not in current_failure_X:
            current_failure_X[column] = 0
    current_failure_X = current_failure_X[failure_results["feature_columns"]]
    current_failure_predictions = failure_results["full_model"].predict(
        current_failure_X
    )
    current_failure_output = pd.DataFrame({
        "pump_id": current_failure_features["pump_id"].to_numpy(),
        "predicted_failure_mode": current_failure_predictions,
    })
    current_failure_output.to_csv(
        OUTPUT_DIR / "current_fleet_failure_mode_predictions.csv",
        index=False,
    )

    history.to_csv(
        OUTPUT_DIR / "history_cleaned_featured.csv.gz",
        index=False,
        compression="gzip",
    )
    current.to_csv(
        OUTPUT_DIR / "current_cleaned_featured.csv.gz",
        index=False,
        compression="gzip",
    )

    quality_report = {
        "project": "Predictive Maintenance - Remaining Useful Life of Industrial Pumps",
        "assumptions": [
            "Maintenance breakdown rows identify pump failure mode.",
            "The maximum observed hour_index is the failure time for each run-to-failure pump.",
            "RUL is defined as failure_hour_index minus current_hour_index.",
            "A 72-hour alert threshold uses a planned repair cost of INR 40,000.",
            "An avoided breakdown is valued at INR 310,000, based on INR 350,000 less planned repair cost.",
            "For the asymmetric score, a prediction above true RUL (late prediction) receives 3x error weight.",
        ],
        "cost_model": {
            "breakdown_cost_inr": BREAKDOWN_COST,
            "planned_repair_cost_inr": PLANNED_REPAIR_COST,
            "cost_avoided_by_planned_repair_inr": BREAKDOWN_COST - PLANNED_REPAIR_COST,
            "target_notice_hours": TARGET_NOTICE_HOURS,
        },
        "history": history_quality,
        "current": current_quality,
        "maintenance": {
            "rows": len(maintenance),
            "pump_count": int(maintenance["pump_id"].nunique()),
            "breakdown_events": int(maintenance["activity"].str.startswith("BREAKDOWN", na=False).sum()),
            "routine_events": int((~maintenance["activity"].str.startswith("BREAKDOWN", na=False)).sum()),
            "total_downtime_hours": int(maintenance["downtime_hours"].sum()),
        },
        "pump_master": {
            "rows": len(pump_master),
            "pump_count": int(pump_master["pump_id"].nunique()),
            "history_pumps_without_master": sorted(set(history["pump_id"]) - set(pump_master["pump_id"])),
            "current_pumps_without_master": sorted(set(current["pump_id"]) - set(pump_master["pump_id"])),
        },
        "validation": {
            "method": "5-fold pump-level GroupKFold for RUL; 5-fold pump-level stratified KFold for failure mode",
            "rule": "No hourly rows from the same pump may appear in training and test folds.",
        },
        "rul_results": {
            key: value
            for key, value in rul_results.items()
            if key not in {
                "oof_predictions",
                "alert_table",
                "full_model",
                "feature_columns",
                "static_columns",
                "history_labeled",
            }
        },
        "failure_mode_results": {
            key: value
            for key, value in failure_results.items()
            if key not in {"oof_predictions", "full_model", "feature_columns"}
        },
        "sequence_model": sequence_results,
    }
    (OUTPUT_DIR / "data_science_report.json").write_text(
        json.dumps(quality_report, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(quality_report, indent=2))
    print(f"\nOutputs written to: {OUTPUT_DIR}")


main()
